# A3.7 · The agent gateway: one choke point when you scale

**Function A — Securing AI Architectures → Securing the Architecture — Runtime and the Gateway**  ·  *Security of AI*

Builds on **[A3.6 · Human approval that survives volume](https://spbreed.github.io/cyber-commons/lessons/A3.6.html)**.

| | |
|---|---|
| Tools used | agentgateway, OPA, Keycloak |

## What this lesson is

**What it covers.** Route every call through one gateway and show the same policy holding for agents that never implemented it.

**Why a security engineer needs it.** Per-agent controls diverge as the fleet grows, and legacy downstreams force a static credential back into agent code. The control it builds is: a single enforcement point holding identity, policy, egress, budget and audit — with the credential for legacy systems held there rather than by the agent.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

At one agent the controls live in the agent. At fifty, each team implements them slightly differently, none of them is audited, and the only honest answer to "is default-deny on?" is "in some of them".

> **At CyberTravels.** At four agents the controls live in the agents. When CyberTravels ships the eighth, nobody can answer “is default-deny on?” with anything better than “in some of them”. R9.

## 2 · The framework

```
   one agent                     fifty agents
   +--------------+              +-----+ +-----+ +-----+ +-----+
   | controls in  |              | a1  | | a2  | | ... | | a50 |
   | the agent    |              +--+--+ +--+--+ +--+--+ +--+--+
   +--------------+                 |       |       |       |
                                    +-------+---+---+-------+
                                                v
                                        +--------------+
                                        |   gateway    | identity, policy,
                                        +------+-------+ budget, egress, log
                                               v
                                            tools

   one place to enforce, one place to audit, one place to turn off
```

**Mitigates: every threat in this chapter, at one enforcement point.**

Everything in Chapters 2 and 3 works. The problem is where it lives.

At one agent, the controls sit in the agent, and that is fine. At fifty, it
stops being fine for reasons that have nothing to do with security engineering:

- Each team implements provenance, budgets and egress slightly differently.
- Nobody can answer "is this control on, everywhere" without reading fifty
  repositories.
- A new agent starts at zero and re-earns every control by hand.
- Fixing a control means fifty pull requests and a migration.

The **gateway** is the same controls, moved to a point every call must pass
through. It holds identity (A2.1–A2.3), policy (A3.1), egress (A3.3), budgets
(A3.4) and audit (A2.7). An agent that implements none of them still gets all of
them, because the enforcement is no longer the agent's responsibility.

It also solves a problem nothing else does: **downstream systems that cannot
consume a delegated identity.** A legacy database or a vendor API that only
understands a static credential forces that credential back into agent code —
undoing A2.3 completely. The gateway holds it instead, authorises the *user*
before the call, and presents the static credential onward. The agent never sees
it.

The honest cost: the gateway is now a single point of failure and a very
attractive target. It has to be operated accordingly.

> **What this control closes.**
>
> Not a new control. The same controls, at a point every call passes through — and the only answer to a downstream that cannot consume delegated identity.

## 3 · Proving every call goes through it, as a skill

A gateway is only a choke point if nothing routes around it, and "nothing routes around it" is not a fact you can read off a route table. The procedure tests reachability from inside the deployment, covers the paths people forget — tool-initiated calls, background jobs, retries — and records each guardrail's **action**, because a filter set to observe is a filter that is switched on and stopping nothing. This is the file in this repository:

In [ ]:
# skills/attestation/llm-gateway-guardrail-verifier/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: llm-gateway-guardrail-verifier
description: >-
  Prove all model traffic leaves through the sanctioned gateway, including
  tool-initiated and background calls, and that guardrail policies are
  attached and enforced. Use to attest gateway routing, to check whether a
  provider endpoint is directly reachable, or to evidence which guardrail
  policies are switched on.
allowed-tools: Bash, Read
---

# Llm Gateway Guardrail Verifier

**Controls:** Control 4 — gateway routing and guardrails

## Confidence: HIGH **only if** egress is enforced below the application

An application-layer gateway configuration is a routing preference. An agent
that can open a socket can bypass it by calling the provider directly. If the
allowlist is not enforced at the network layer, **downgrade this control to
PARTIAL** and say why.

## Procedure

1. **Test reachability, do not read configuration.** From the deployment's
   network position, attempt to reach provider endpoints directly. Anything
   reachable that is not the gateway is a finding, regardless of what the
   configuration says.

2. **Cover the paths people forget.** Tool-initiated calls, background jobs,
   scheduled tasks, retry paths and sub-agents. A gateway that fronts the main
   request path and not the batch job is a gateway with a hole in it.

3. **Confirm guardrail attachment and version.** Record the guardrail ID and
   version actually attached to the route, not the one in the template.

4. **Enumerate enabled policies.** Content filters, prompt-attack filtering,
   denied topics, sensitive-information filters, contextual grounding. Record
   whether each is applied on **input, output, or both** — input-only filtering
   is a common and quiet gap.

5. **Confirm the action.** A filter set to observe rather than block is
   telemetry, not a control. Record the configured action per policy.

## Output contract

```json
{
  "deployment_id": "str",
  "gateway_enforced": true,
  "enforcement_layer": "network|application",
  "reachable_provider_findings": [{"endpoint": "str", "path": "direct|tool|background"}],
  "guardrail": {"id": "str", "version": "str",
                "policies": [{"name": "str", "applied_to": "input|output|both",
                              "action": "block|anonymize|observe"}]},
  "verdict": "PASS|PARTIAL|FAIL"
}
```

## Failure modes

- **Reading the route table instead of testing reachability.**
- **Missing the background path.** Scheduled and tool-initiated calls are the
  ones that bypass the gateway in practice.
- **Recording a guardrail as enforced when its action is observe.**
"""

In [ ]:
import json, re

def parse_skill(md):
    """Split a SKILL.md into (frontmatter dict, body).

    Frontmatter is a small, fixed subset of YAML: `key: value`, plus folded
    scalars (`description: >-`) whose continuation lines are indented. That is
    all a skill needs, and parsing it directly means no dependency.
    """
    if not md.startswith("---"):
        raise ValueError("a SKILL.md must open with a frontmatter block")
    _, front, body = md.split("---", 2)
    meta, key = {}, None
    for line in front.strip().splitlines():
        if not line.strip():
            continue
        if not line[0].isspace() and ":" in line:
            key, val = line.split(":", 1)
            key, val = key.strip(), val.strip()
            # `>-` and `|` open a folded block; the value is on the next lines
            meta[key] = "" if val in (">-", ">", "|", "|-") else val
        elif key is not None:
            meta[key] = (meta[key] + " " + line.strip()).strip()
    if "allowed-tools" in meta:
        meta["allowed-tools"] = [t.strip() for t in meta["allowed-tools"].split(",")
                                 if t.strip()]
    for required in ("name", "description"):
        if not meta.get(required):
            raise ValueError(f"skill is missing a {required!r}")
    return meta, body.strip()

_WORD = re.compile(r"[a-z][a-z-]{3,}")

def route(task, skills):
    """Pick the skill whose description best matches a task. Deterministic.

    The description is not documentation — it is the routing key. An agent
    decides whether to load a skill by reading it, so a vague description means
    the skill never fires when it should, and two overlapping descriptions mean
    the wrong one fires.

    Returns (pick, scores, margin). A margin of 0 means the top two scored the
    same and the "winner" is just whichever sorted first — an arbitrary answer
    wearing a confident face. Callers should refuse to auto-route on margin 0
    rather than pretend the tiebreak meant something.
    """
    want = set(_WORD.findall(task.lower()))
    def score(meta):
        return len(want & set(_WORD.findall(meta["description"].lower())))
    scores = {n: score(skills[n]) for n in sorted(skills)}
    # sort names first, then by score: ties must break identically on every
    # machine or the same task routes differently on two runs
    ranked = sorted(sorted(skills), key=lambda n: -scores[n])
    top = scores[ranked[0]]
    margin = top - (scores[ranked[1]] if len(ranked) > 1 else 0)
    return ranked[0], scores, margin

def contract_of(body):
    """The JSON block under '## Output contract' — the skill's machine promise."""
    # non-greedy across any prose between the heading and the fence
    m = re.search(r"## Output contract\b.*?```json\n(.*?)```", body, re.S)
    if not m:
        raise ValueError("skill declares no output contract")
    return json.loads(m.group(1))

def check(instance, contract, path="$"):
    """Structural conformance of an instance against a contract template.

    Returns the list of problems. An empty list means the shape is right — and
    that is *all* it means. Conformance is not accuracy: an empty findings list
    conforms perfectly and tells you nothing.
    """
    problems = []
    if isinstance(contract, dict):
        if not isinstance(instance, dict):
            return [f"{path}: expected an object, got {type(instance).__name__}"]
        for k, v in sorted(contract.items()):
            if k not in instance:
                problems.append(f"{path}.{k}: missing")
            else:
                problems += check(instance[k], v, f"{path}.{k}")
    elif isinstance(contract, list):
        if not isinstance(instance, list):
            return [f"{path}: expected a list, got {type(instance).__name__}"]
        for i, item in enumerate(instance):          # every element, same template
            problems += check(item, contract[0], f"{path}[{i}]")
    elif isinstance(contract, str) and "|" in contract:
        if instance not in contract.split("|"):
            problems.append(f"{path}: {instance!r} is not one of {contract}")
    elif isinstance(contract, bool):                  # before the numeric case:
        if not isinstance(instance, bool):            # bool is a subclass of int
            problems.append(f"{path}: expected bool, got {type(instance).__name__}")
    elif isinstance(contract, (int, float)):
        # JSON has one number type. A contract written `0` must accept 0.4, or
        # every cost and rate in the pipeline has to be rounded to satisfy a
        # checker rather than to be correct.
        if isinstance(instance, bool) or not isinstance(instance, (int, float)):
            problems.append(f"{path}: expected a number, got {type(instance).__name__}")
    elif not isinstance(instance, type(contract)):
        problems.append(f"{path}: expected {type(contract).__name__}, "
                        f"got {type(instance).__name__}")
    return problems

# Execute the skill above: parse skills/attestation/llm-gateway-guardrail-verifier/SKILL.md into the two
# halves an agent uses — the frontmatter it routes on, and the body
# it follows.
meta, body = parse_skill(SKILL_MD)
print(f"loaded skill: {meta['name']}")
print(f"  tools it may use: {', '.join(meta.get('allowed-tools', [])) or '—'}")
print(f"  routing description: {len(meta['description'].split())} words")
print(f"  procedure: {len(body.splitlines())} lines")

In [ ]:
# skills/attestation/llm-gateway-guardrail-verifier/scripts/llm_gateway_guardrail_verifier.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Send five calls through one gateway and record which check refuses each one.

This is the executable half of the `llm-gateway-guardrail-verifier` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

LEGACY_DB_CREDENTIAL = "static-service-password"     # never leaves the gateway

REGISTRY = {"spiffe://corp/reports-agent": {"owner": "sam@corp", "expires": 9000}}
POLICY = {("reports-agent", "run_query", "table:reports"): {"SELECT"}}
EGRESS_ALLOW = {"reports-db.corp.example"}

AUDIT = []

def gateway(call):
    """One choke point: identity, registry, policy, egress, budget, audit."""
    checks = []
    def check(name, ok, why=""):
        checks.append((name, ok, why)); return ok

    if not check("identity", call["identity"] in REGISTRY, "attested and registered"):
        return {"allowed": False, "checks": checks}
    verbs = POLICY.get((call["agent"], call["tool"], call["resource"]), set())
    if not check("policy", call["verb"] in verbs, f"permitted verbs {sorted(verbs) or 'none'}"):
        return {"allowed": False, "checks": checks}
    if not check("egress", call["destination"] in EGRESS_ALLOW, "destination allow-list"):
        return {"allowed": False, "checks": checks}
    if not check("budget", call["calls_so_far"] < 5, "per-target ceiling"):
        return {"allowed": False, "checks": checks}

    # the agent never held this; the gateway attaches it on the way out
    AUDIT.append({"principal": call["principal"], "agent": call["agent"],
                  "tool": call["tool"], "resource": call["resource"]})
    return {"allowed": True, "checks": checks, "credential_attached": LEGACY_DB_CREDENTIAL[:6] + "..."}

BASE = {"identity": "spiffe://corp/reports-agent", "agent": "reports-agent",
        "principal": "dana@corp", "tool": "run_query", "resource": "table:reports",
        "verb": "SELECT", "destination": "reports-db.corp.example", "calls_so_far": 0}

CASES = {
 "the intended call":          BASE,
 "unregistered agent":         dict(BASE, identity="spiffe://corp/rogue-agent"),
 "verb not permitted":         dict(BASE, verb="DELETE"),
 "exfiltration destination":   dict(BASE, destination="archive.evil.example"),
 "over the per-target ceiling":dict(BASE, calls_so_far=9),
}
for label, call in CASES.items():
    r = gateway(call)
    failed = [n for n, ok, _ in r["checks"] if not ok]
    print(f"   {label:28s}{'ALLOWED' if r['allowed'] else 'denied at ' + failed[0]}")

print(f"\naudit entries written: {len(AUDIT)}")
print(f"credential held by the agent: never - attached at the gateway")
print()
print("The agent implements none of this. Add a new agent tomorrow and it")
print("inherits every control by being on the other side of one hop.")
print()
print("And the legacy database, which cannot consume a delegated token, is")
print("reached with a static credential the agent has never seen - authorised")
print("against dana before the call was made.")
assert len(AUDIT) == 1 and gateway(CASES["unregistered agent"])["allowed"] is False

## What you just proved

The skill loads and reports its shape. Its confidence is HIGH only where egress is enforced below the application — the gateway is a choke point because the network makes it one, not because the SDK was configured to point at it, and an application-level base URL is a default, not a control.

## Your turn

Count your agents. If it is more than five, work out how you would currently answer 'is egress control on for all of them' — and how long that would take.

---

**Next → [A3.8 · Shared infrastructure between agent runs](https://spbreed.github.io/cyber-commons/lessons/A3.8.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A3.7.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A3.7.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*